# Project Work

## Model 2: Pretrained BERT MCQ Solver

This notebook fine-tunes a local Hugging Face multiple-choice classifier and keeps the flow aligned to the viva walkthrough: setup, data pipeline, architecture, training/evaluation, and submission.


# 1. Setup & Configuration

## 1.1 Library Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Imports ready.')

## 1.2 BERT Model Download and Cache


In [ ]:
# Clone bert-base-uncased locally to avoid Kaggle download timeouts
MODEL_DIR = './bert-base-uncased'

if not os.path.exists(MODEL_DIR):
    print('Cloning bert-base-uncased...')
    os.system('git clone https://huggingface.co/bert-base-uncased')
    print('Clone complete.')
else:
    print(f'Using cached model at {MODEL_DIR}')

print(os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else 'Clone may have failed.')

## 1.3 Path Resolution and Random Seed


In [ ]:
# then local project paths, so the same notebook runs on Kaggle and offline.
KAGGLE_INPUT = '/kaggle/input/competitions/smart-mcq-solver-challenge'

def get_path(filename: str) -> str:
    candidates = [
        os.path.join(KAGGLE_INPUT, filename),
        os.path.join('..', '..', 'data', filename),
        os.path.join('data', filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)
print(f'Device : {DEVICE}')


## 1.4 W&B Configuration and Hyperparameters


In [ ]:
# Safe W&B Secret Handling for Kaggle & Local Execution
wandb_mode = 'disabled'
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key    = user_secrets.get_secret('WANDB_API_KEY')
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        wandb_mode = 'online'
except Exception:
    # Fallback for environments without Kaggle Secrets attached
    env_key = os.environ.get('WANDB_API_KEY')
    if env_key:
        wandb.login(key=env_key, relogin=True)
        wandb_mode = 'online'
    else:
        print(f"W&B key not detected. Running notebook in '{wandb_mode}' mode.")

# Initialize run cleanly without blocking
run = wandb.init(
    project='23f2004343-t22026',
    name='Model_2_BERT_MultipleChoice',
    mode=wandb_mode,
    config={
        'base_model'    : 'bert-base-uncased',
        'max_seq_length': 128,
        'batch_size'    : 8,
        'epochs'        : 3,
        'learning_rate' : 2e-5,
        'warmup_ratio'  : 0.1,
        'weight_decay'  : 0.01,
        'optimizer'     : 'AdamW',
        'loss_fn'       : 'CrossEntropyLoss',
        'num_options'   : 5,
    },
)
cfg = wandb.config
print(f'W&B run : {run.name}  |  mode : {wandb_mode}  |  project : {run.project}')
print(f'Config  : {dict(cfg)}')


# 2. Data Pipeline


## 2.1 CSV Loading and Tokenizer Initialization


In [ ]:
trn_df = pd.read_csv(get_path('train.csv'))
tst_df = pd.read_csv(get_path('test.csv'))
print(f'train : {trn_df.shape}  |  test : {tst_df.shape}')
print(f'columns : {list(trn_df.columns)}')

OPTIONS = ['A', 'B', 'C', 'D', 'E']

# Load tokenizer from local clone; hub name is the fallback
MODEL_NAME = MODEL_DIR if os.path.exists(MODEL_DIR) else cfg.base_model
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = int(cfg.max_seq_length)
print(f'Tokenizer ready  |  max_len={MAX_LEN}')

## 2.2 MCQ Dataset Class and Input Construction


In [ ]:
class MCQMultiChoiceDataset(Dataset):
    """Tokenizes each MCQ row into 5 (prompt, option) pair encodings.

    Output tensors per item:
      input_ids      : (5, max_len)
      attention_mask : (5, max_len)
      label          : int in [0, 4] — correct option index (train only)
    """

    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        prompt = str(row['prompt'])

        # Tokenize each (prompt, option) pair — output shape: (5, max_len)
        encodings = tokenizer(
            [prompt] * 5,
            [str(row[opt]) for opt in OPTIONS],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        # pooling and self-attention do not treat padding as meaningful text.
        item = {
            'input_ids'     : encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
        }

        if self.is_train:
            ans           = str(row['answer']).strip().upper()
            label         = OPTIONS.index(ans) if ans in OPTIONS else 0
            item['label'] = torch.tensor(label, dtype=torch.long)

        return item

print('Dataset class defined.')


## 2.3 Train/Validation Split and DataLoaders


In [ ]:
# 90/10 train-validation split, reproducible via fixed seed
trn_sub, val_sub = train_test_split(trn_df, test_size=0.1, random_state=SEED)

BATCH = int(cfg.batch_size)

trn_loader = DataLoader(
    MCQMultiChoiceDataset(trn_sub, is_train=True),
    batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    MCQMultiChoiceDataset(val_sub, is_train=True),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)
tst_loader = DataLoader(
    MCQMultiChoiceDataset(tst_df, is_train=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)

probe = next(iter(trn_loader))
print(f"input_ids shape : {probe['input_ids'].shape}   # (batch, 5, max_len)")
print(f"labels sample   : {probe['label'].tolist()[:8]}")

# 3. Model Architecture & Definition


## 3.1 BERT MultipleChoice Model and Optimizer Setup


In [ ]:
# AutoModelForMultipleChoice: takes (batch, 5, seq_len), returns logits (batch, 5)
# BERT runs once per option internally, linear head scores the [CLS] token
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model      : {cfg.base_model}')
print(f'Parameters : {n_params:,} trainable')

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=float(cfg.learning_rate),
    weight_decay=float(cfg.weight_decay),
    eps=1e-8,
)

total_steps  = len(trn_loader) * int(cfg.epochs)
warmup_steps = int(total_steps * float(cfg.warmup_ratio))
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Steps : {total_steps}  |  warmup : {warmup_steps}')

# 4. Training and Evaluation


## 4.1 Metric Functions: Loss, Accuracy, F1, and MAP@3


In [ ]:
# the top-three ranked labels, matching the Kaggle scoring rule.
def ap_at_3(ranked: list, correct: int) -> float:
    """Average Precision at 3 for one question.

    MAP@3 rewards the correct answer most when it appears first in the top-three ranking.
    """
    hits, score = 0, 0.0
    for k, pred in enumerate(ranked[:3], start=1):
        if pred == correct:
            hits  += 1
            score += hits / k
    return score

def run_eval(model, loader) -> dict:
    """Full validation pass — returns loss, accuracy, macro-F1, and MAP@3.

    Viva note: MAP@3 is logged for the competition objective, while val_loss is used for stable checkpointing.
    """
    model.eval()
    total_loss, all_preds, all_labels, all_ap3 = 0.0, [], [], []

    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            labs  = batch['label'].to(DEVICE)

            out    = model(input_ids=ids, attention_mask=masks)
            logits = out.logits  # (batch, 5) — raw scores before softmax
            loss   = criterion(logits, labs)
            total_loss += loss.item()

            # Extract top 3 highest-scoring options for MAP@3 scoring
            top_3  = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().tolist()
            preds  = [t[0] for t in top_3]   # top-1 prediction for accuracy/F1
            labels = labs.cpu().tolist()

            all_preds.extend(preds)
            all_labels.extend(labels)

            # Compute AP@3 per question in the batch
            for ranked, correct in zip(top_3, labels):
                all_ap3.append(ap_at_3(ranked, correct))

    n = max(len(loader), 1)
    return {
        'val_loss'    : total_loss / n,
        'val_accuracy': accuracy_score(all_labels, all_preds),
        'val_f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'val_map3'    : float(np.mean(all_ap3)) if all_ap3 else 0.0,
    }


## 4.2 Training Loop and Checkpointing


In [ ]:
EPOCHS        = int(cfg.epochs)
CKPT_PATH     = '/kaggle/working/bert_mcq_best.pt'
best_val_loss = float('inf')
best_val_map3 = 0.0   # running tracker — passive, not used for early stopping

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in trn_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)
        labs  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=masks).logits
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    avg_trn = running_loss / max(len(trn_loader), 1)
    val_m   = run_eval(model, val_loader)

    # Keep track of best MAP@3 seen so far — for W&B summary only
    best_val_map3 = max(best_val_map3, val_m['val_map3'])

    wandb.log({'epoch': epoch, 'train_loss': avg_trn, **val_m})
    print(
        f'Epoch {epoch}/{EPOCHS}  '
        f'trn={avg_trn:.4f}  '
        f'val_loss={val_m["val_loss"]:.4f}  '
        f'val_acc={val_m["val_accuracy"]:.4f}  '
        f'val_f1={val_m["val_f1_macro"]:.4f}  '
        f'val_map3={val_m["val_map3"]:.4f}'
    )

    # fluctuate early in training; loss convergence is a more stable signal
    if val_m['val_loss'] < best_val_loss:
        best_val_loss = val_m['val_loss']
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'  ✓ Checkpoint saved  val_loss={best_val_loss:.4f}')

wandb.run.summary['best_val_loss'] = best_val_loss
wandb.run.summary['best_val_map3'] = best_val_map3
print('\nTraining complete.')


# 5. Inference & Submission


## 5.1 Checkpoint Reload and Batched Inference


In [ ]:
# Load best checkpoint saved during training
if os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    print(f'Loaded checkpoint: {CKPT_PATH}')

model.eval()
OPTIONS_ARR = np.array(OPTIONS)
all_top3    = []

print(f'Running batched inference on {len(tst_df)} test questions...')
with torch.no_grad():
    for batch in tst_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)

        logits   = model(input_ids=ids, attention_mask=masks).logits  # (B, 5)
        top3_idx = torch.topk(logits, 3, dim=1).indices.cpu().numpy()  # (B, 3) descending

        for row_idx in top3_idx:
            all_top3.append(' '.join(OPTIONS_ARR[row_idx]))

# the exact Kaggle-required schema: ID and Prediction.
# Align output with sample_submission.csv column names to prevent KeyError
sub_template = pd.read_csv(get_path('sample_submission.csv'))
id_col, pred_col = sub_template.columns[0], sub_template.columns[1]
print(f'Template columns : {[id_col, pred_col]}')
print(f'Predictions      : {len(all_top3)}')

assert len(all_top3) == len(tst_df), \
    f'Row count mismatch: expected {len(tst_df)}, got {len(all_top3)}'

sub_df = pd.DataFrame({id_col: tst_df['id'].values, pred_col: all_top3})
sub_df.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub_df)} rows')
print(sub_df.head())

# Log final artifact and close W&B run
art = wandb.Artifact('submission_bert_mcq', type='predictions')
art.add_file('submission.csv')
wandb.log_artifact(art)
wandb.finish()
print('W&B run closed.')
